### 1) Custom knowledge base
#### dspy.pdf

### 2) Set up Asyncio

In [12]:
import nest_asyncio

nest_asyncio.apply()

### 3) Set up the Qdrant vector database

In [13]:
from qdrant_client import QdrantClient

collection_name = "chat_with_docs"

client = QdrantClient(
    host="localhost",
    port="6333"
)

### 4) Read the documents

In [14]:
from llama_index.core import SimpleDirectoryReader

input_dir_path = './docs'

loader = SimpleDirectoryReader(
    input_dir=input_dir_path,
    required_exts=[".pdf"],
    recursive=True
)
docs = loader.load_data()

2026-05-18 01:09:19,510 - INFO - HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


In [15]:
type(docs), len(docs)

(list, 32)

### 5) A function to index data

In [16]:
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex, StorageContext

def create_index(documents):
    vector_store = QdrantVectorStore(client=client, collection_name=collection_name)

    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)

    return index

### 6) Load the embedding model and index data

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("HF_TOKEN"), "HF_TOKEN is not set. Copy .env.example to .env and fill in your token."


In [18]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-large-en-v1.5", trust_remote_code=True)

Settings.embed_model = embed_model

index = create_index(docs)

2026-05-18 01:09:20,604 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-18 01:09:20,617 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-05-18 01:09:20,887 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-18 01:09:20,896 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-18 01:09:20,897 - INFO - Loading SentenceTransformer model from BAAI/bge-large-en-v1.5.
2026-05-18 01:09:21,181 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 3

### 7) Load the LLM

In [19]:
from llama_index.llms.ollama import Ollama

llm = Ollama(model="llama3.2", request_timeout=120.0)

Settings.llm = llm

### 8) Define the prompt template

In [20]:
from llama_index.core import PromptTemplate

template = """Context information is below:
              ---------------------
              {context_str}
              ---------------------
              Given the context information above I want you to think
              step by step to answer the query in a crisp manner,
              incase you don't know the answer say 'I don't know!'
            
              Query: {query_str}
        
              Answer:"""

qa_prompt_tmpl = PromptTemplate(template)

### 9) Reranking

In [21]:
from llama_index.core.postprocessor import SentenceTransformerRerank

rerank = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-2-v2", 
    top_n=3
)

2026-05-18 01:09:41,398 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-2-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-18 01:09:41,677 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L2-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-05-18 01:09:41,683 - INFO - No modules.json found for cross-encoder/ms-marco-MiniLM-L-2-v2, initializing a new CrossEncoder model.
2026-05-18 01:09:41,961 - INFO - HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-2-v2 "HTTP/1.1 307 Temporary Redirect"
2026-05-18 01:09:42,240 - INFO - HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L2-v2 "HTTP/1.1 200 OK"
2026-05-18 01:09:42,512 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-2-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-18 01:09:42,780 - INFO - HTTP Request: HEAD https://hugg

### 10) Query the document

In [22]:
query_engine = index.as_query_engine(similarity_top_k=10,
                                     node_postprocessors=[rerank])

query_engine.update_prompts(
    {"response_synthesizer:text_qa_template": qa_prompt_tmpl}
)

response = query_engine.query("What exactly is DSPy?")

2026-05-18 01:09:53,689 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"
2026-05-18 01:09:54,921 - INFO - HTTP Request: POST http://localhost:6333/collections/chat_with_docs/points/query "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00,  2.44it/s]
2026-05-18 01:10:04,901 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


In [23]:
from IPython.display import Markdown, display

display(Markdown(str(response)))

DSPy (Dee-ess-pie) is a programming model that translates prompting techniques into parameterized declarative modules. It's designed to make it easier to create self-improving and pipeline-adaptive systems by providing a way to abstractly specify what a text transformation needs to do, rather than how a specific LM should be prompted to implement that behavior.

# Limitations  !!!

### 1. How to deal with complex documents that include tables and figures.
### 2. How to finetune embedding and reranker model for domain-specific use case?
### 3. How to make RAG faster to scale over millions of vectors?
### 4. How to build multimodal RAG, where the additional knowledge base has both text and images?
### 5. How to evaluate your RAG app?
### 6. How to build observability measures in your RAG to ensure that it's working well in production?